In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [8]:
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph(
    url="neo4j+s://fa687069.databases.neo4j.io",
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="fa687069",
    refresh_schema=False
)
graph

In [9]:
movie_query = """
LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/tomasonjo/blog-datasets/refs/heads/main/movies/movies_small.csv' AS row

MERGE (m:Movie {id: row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)

WITH m, row
UNWIND split(row.director, '|') AS director
MERGE (d:Person {name: trim(director)})
MERGE (d)-[:DIRECTED]->(m)

WITH m, row
UNWIND split(row.actors, '|') AS actor
MERGE (a:Person {name: trim(actor)})
MERGE (a)-[:ACTED_IN]->(m)

WITH m, row
UNWIND split(row.genres, '|') AS genre
MERGE (g:Genre {name: trim(genre)})
MERGE (m)-[:IN_GENRE]->(g)
"""

In [10]:
graph._driver.session(database=graph._database).run(movie_query)

In [11]:
graph.refresh_schema()

In [12]:
graph.get_schema

'Node properties:\nMovie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}\nPerson {name: STRING}\nGenre {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Movie)-[:IN_GENRE]->(:Genre)\n(:Person)-[:DIRECTED]->(:Movie)\n(:Person)-[:ACTED_IN]->(:Movie)'